# 03 · SQL Analysis in PostgreSQL

**Goal:** answer the business questions with SQL running directly against the PostgreSQL database
built by `src/build_db.py`.

Every query lives in its own file in `sql/`, so it can be reviewed, reused or run in pgAdmin.
This notebook executes them and interprets the results.

| # | File | SQL techniques |
| --- | --- | --- |
| 1 | `01_monthly_revenue.sql` | CTE, `DATE_TRUNC`, aggregation |
| 2 | `02_mom_growth.sql` | `LAG()` window function |
| 3 | `03_top_categories.sql` | 3-table `JOIN`, `RANK()`, running share with `SUM() OVER` |
| 4 | `04_revenue_by_state.sql` | share of total with `SUM(SUM()) OVER ()` |
| 5 | `05_repeat_customers.sql` | CTE, `COUNT(*) FILTER (WHERE ...)` |
| 6 | `06_cohorts.sql` | multi-step CTE, `MIN() OVER (PARTITION BY)`, `AGE()` |
| 7 | `07_rfm_base.sql` | `NTILE(5)`, scalar subquery, `CASE` |
| 8 | `08_delivery_vs_review.sql` | `CASE` buckets, `FILTER` |
| 9 | `09_running_total.sql` | cumulative and YTD sums, `LAG(…, 12)` |
| 10 | `10_payment_types.sql` | `PERCENTILE_CONT` (median) |

**Definitions** (from `02_cleaning.ipynb`): revenue = `price + freight_value`; customer = `customer_unique_id`;
trend period = Jan 2017 – Aug 2018; all amounts in Brazilian reais (BRL).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

sys.path.append(str(Path("..").resolve()))
from src.db import SQL_DIR, get_engine, run_sql_file

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 40)

engine = get_engine()


def show_query(filename):
    """Display the SQL text of a query file."""
    sql = (SQL_DIR / filename).read_text(encoding="utf-8")
    display(Markdown(f"```sql\n{sql}\n```"))

In [2]:
# Sanity check: the database contains the expected tables
tables = ["customers", "products", "orders", "order_items", "order_payments", "order_reviews"]
pd.read_sql(
    " UNION ALL ".join(f"SELECT '{t}' AS table_name, COUNT(*) AS rows FROM {t}" for t in tables),
    engine,
)

,table_name,rows
0,products,32214
1,customers,96470
2,order_items,110189
3,order_reviews,95824
4,order_payments,100748
5,orders,96470


## Q1. How did revenue, orders and AOV change by month?

In [3]:
monthly = run_sql_file(engine, "01_monthly_revenue.sql")
show_query("01_monthly_revenue.sql")
monthly

```sql
-- Q1. Monthly revenue, orders and average order value (AOV)
-- Revenue = price + freight_value. Trend period: Jan 2017 – Aug 2018.
WITH order_revenue AS (
    SELECT
        o.order_id,
        DATE_TRUNC('month', o.order_purchase_timestamp)::date AS month,
        SUM(oi.price + oi.freight_value)                       AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY o.order_id, month
)
SELECT
    month,
    COUNT(*)                      AS orders,
    ROUND(SUM(revenue), 2)        AS revenue,
    ROUND(AVG(revenue), 2)        AS aov
FROM order_revenue
GROUP BY month
ORDER BY month;

```

,month,orders,revenue,aov
0,2017-01-01,750,"127,482.37",169.98
1,2017-02-01,1653,"271,239.32",164.09
2,2017-03-01,2546,"414,330.95",162.74
3,2017-04-01,2303,"390,812.40",169.70
4,2017-05-01,3545,"566,657.40",159.85
5,2017-06-01,3135,"490,050.37",156.32
6,2017-07-01,3872,"566,299.08",146.25
7,2017-08-01,4193,"645,832.36",154.03
8,2017-09-01,4150,"701,077.49",168.93
9,2017-10-01,4478,"751,117.01",167.73


In [4]:
first, last = monthly.iloc[:3], monthly.iloc[-3:]
print(f"Total revenue (Jan 2017 – Aug 2018): {monthly['revenue'].sum():,.0f} BRL")
print(f"Orders per month, first 3 months avg: {first['orders'].mean():,.0f}")
print(f"Orders per month, last 3 months avg:  {last['orders'].mean():,.0f}")
print(f"AOV range: {monthly['aov'].min():.0f} – {monthly['aov'].max():.0f} BRL")
best = monthly.loc[monthly['revenue'].idxmax()]
print(f"Best month: {best['month']:%b %Y} with {best['revenue']:,.0f} BRL")

Total revenue (Jan 2017 – Aug 2018): 15,371,741 BRL
Orders per month, first 3 months avg: 1,650
Orders per month, last 3 months avg:  6,201
AOV range: 146 – 170 BRL
Best month: Nov 2017 with 1,153,229 BRL


**Insights**
- The business grew fast during 2017: monthly orders went from ~1,650 to ~6,600.
- Growth came from **more orders, not bigger baskets**: AOV stayed in a narrow 146–170 BRL band.
- **November 2017 is the peak month** (Black Friday), followed by a dip in December.
- From March 2018 revenue **plateaus at ~1.0–1.1 M BRL per month**: the growth phase is over.

## Q2. Month-over-month growth

In [5]:
mom = run_sql_file(engine, "02_mom_growth.sql")
show_query("02_mom_growth.sql")
mom

```sql
-- Q2. Month-over-month revenue growth with LAG()
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp)::date AS month,
        SUM(oi.price + oi.freight_value)                       AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY month
)
SELECT
    month,
    ROUND(revenue, 2)                                        AS revenue,
    ROUND(LAG(revenue) OVER (ORDER BY month), 2)             AS prev_month_revenue,
    ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
          / LAG(revenue) OVER (ORDER BY month), 1)           AS mom_growth_pct
FROM monthly
ORDER BY month;

```

,month,revenue,prev_month_revenue,mom_growth_pct
0,2017-01-01,"127,482.37",NaN,NaN
1,2017-02-01,"271,239.32","127,482.37",112.80
2,2017-03-01,"414,330.95","271,239.32",52.80
3,2017-04-01,"390,812.40","414,330.95",-5.70
4,2017-05-01,"566,657.40","390,812.40",45.00
5,2017-06-01,"490,050.37","566,657.40",-13.50
6,2017-07-01,"566,299.08","490,050.37",15.60
7,2017-08-01,"645,832.36","566,299.08",14.00
8,2017-09-01,"701,077.49","645,832.36",8.60
9,2017-10-01,"751,117.01","701,077.49",7.10


In [6]:
h1_2017 = mom[(mom["month"] >= pd.Timestamp("2017-02-01").date()) & (mom["month"] <= pd.Timestamp("2017-10-01").date())]
y2018 = mom[mom["month"] >= pd.Timestamp("2018-03-01").date()]
print(f"Average MoM growth Feb–Oct 2017: {h1_2017['mom_growth_pct'].mean():.1f}%")
print(f"Average MoM growth Mar–Aug 2018: {y2018['mom_growth_pct'].mean():.1f}%")

Average MoM growth Feb–Oct 2017: 26.3%
Average MoM growth Mar–Aug 2018: 0.6%


**Insights**
- 2017 shows strong double-digit monthly growth, while 2018 growth is around zero.
- The +53.5% in Nov 2017 and −26.9% in Dec 2017 are a seasonal spike, not a trend change.
  Comparing a month with the same month a year earlier (Q9) removes this seasonality.

## Q3. Which categories generate the most revenue?

In [7]:
categories = run_sql_file(engine, "03_top_categories.sql")
show_query("03_top_categories.sql")
categories

```sql
-- Q3. Top 15 product categories by revenue, ranked with RANK()
WITH category_revenue AS (
    SELECT
        p.category,
        COUNT(DISTINCT oi.order_id)        AS orders,
        SUM(oi.price + oi.freight_value)   AS revenue,
        AVG(oi.price)                      AS avg_item_price
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    JOIN orders   o ON o.order_id   = oi.order_id
    GROUP BY p.category
)
SELECT
    RANK() OVER (ORDER BY revenue DESC)                           AS rank,
    category,
    orders,
    ROUND(revenue, 2)                                             AS revenue,
    ROUND(100.0 * revenue / SUM(revenue) OVER (), 1)              AS revenue_share_pct,
    ROUND(100.0 * SUM(revenue) OVER (ORDER BY revenue DESC)
          / SUM(revenue) OVER (), 1)                              AS cumulative_share_pct,
    ROUND(avg_item_price, 2)                                      AS avg_item_price
FROM category_revenue
ORDER BY revenue DESC
LIMIT 15;

```

,rank,category,orders,revenue,revenue_share_pct,cumulative_share_pct,avg_item_price
0,1,health_beauty,8647,"1,412,089.53",9.20,9.20,130.28
1,2,watches_gifts,5493,"1,264,016.98",8.20,17.40,199.06
2,3,bed_bath_table,9272,"1,225,209.26",7.90,25.30,93.44
3,4,sports_leisure,7529,"1,118,062.91",7.30,32.60,113.25
4,5,computers_accessories,6529,"1,032,603.65",6.70,39.30,116.27
5,6,furniture_decor,6307,"880,329.92",5.70,45.00,87.25
6,7,housewares,5743,"758,392.25",4.90,49.90,90.60
7,8,cool_stuff,3559,"691,680.89",4.50,54.40,164.12
8,9,auto,3809,"669,319.92",4.30,58.70,139.85
9,10,garden_tools,3448,"567,145.68",3.70,62.40,110.24


In [8]:
top5_share = categories.head(5)["revenue_share_pct"].sum()
n_total = pd.read_sql("SELECT COUNT(DISTINCT category) FROM products", engine).iloc[0, 0]
print(f"Top 5 of {n_total} categories = {top5_share:.1f}% of revenue")
print(f"Top 15 categories = {categories['cumulative_share_pct'].iloc[-1]:.1f}% of revenue")

Top 5 of 74 categories = 39.3% of revenue
Top 15 categories = 76.5% of revenue


**Insights**
- Revenue is **concentrated**: 5 of 74 categories bring ~39% of revenue, 15 categories bring ~77%.
- `health_beauty` leads by revenue, while `bed_bath_table` has the most orders but a low item price (93 BRL).
- `watches_gifts` is #2 with fewer orders thanks to the highest average price (199 BRL):
  a candidate for premium promotions.

## Q4. Which states generate the most revenue?

In [9]:
states = run_sql_file(engine, "04_revenue_by_state.sql")
show_query("04_revenue_by_state.sql")
states

```sql
-- Q4. Revenue, customers and AOV by customer state
WITH order_revenue AS (
    SELECT
        o.order_id,
        c.customer_state,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value) AS revenue,
        SUM(oi.freight_value)            AS freight
    FROM orders o
    JOIN customers   c  ON c.customer_id = o.customer_id
    JOIN order_items oi ON oi.order_id   = o.order_id
    GROUP BY o.order_id, c.customer_state, c.customer_unique_id
)
SELECT
    customer_state                                         AS state,
    COUNT(DISTINCT customer_unique_id)                     AS customers,
    COUNT(*)                                               AS orders,
    ROUND(SUM(revenue), 2)                                 AS revenue,
    ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 1) AS revenue_share_pct,
    ROUND(AVG(revenue), 2)                                 AS aov,
    ROUND(100.0 * SUM(freight) / SUM(revenue), 1)          AS freight_share_pct
FROM order_revenue
GROUP BY customer_state
ORDER BY revenue DESC;

```

,state,customers,orders,revenue,revenue_share_pct,aov,freight_share_pct
0,SP,39149,40494,"5,768,518.23",37.40,142.45,12.20
1,RJ,11917,12350,"2,055,401.57",13.30,166.43,14.40
2,MG,11001,11354,"1,818,891.67",11.80,160.20,14.60
3,RS,5167,5344,"861,278.79",5.60,161.17,15.40
4,PR,4769,4923,"781,708.80",5.10,158.79,14.80
5,SC,3449,3546,"595,127.78",3.90,167.83,14.80
6,BA,3158,3256,"591,137.81",3.80,181.55,16.50
7,DF,2019,2080,"346,123.35",2.20,166.41,14.30
8,GO,1895,1957,"334,212.35",2.20,170.78,15.40
9,ES,1928,1995,"317,657.93",2.10,159.23,15.40


In [10]:
top3 = states.head(3)
print(f"SP + RJ + MG = {top3['revenue_share_pct'].sum():.1f}% of revenue")
print(f"Freight share in SP: {states.iloc[0]['freight_share_pct']:.1f}% vs. "
      f"max {states['freight_share_pct'].max():.1f}% ({states.loc[states['freight_share_pct'].idxmax(), 'state']})")

SP + RJ + MG = 62.5% of revenue
Freight share in SP: 12.2% vs. max 21.9% (RR)


**Insights**
- **São Paulo alone is 37% of revenue**; together with Rio de Janeiro and Minas Gerais it is ~63%.
- Customers in remote northern and north-eastern states place **higher-value orders** (AOV 200–270 BRL vs. 142 in SP),
  but pay much more for delivery: freight is up to ~22% of the order value (RR) vs. 12% in SP.
- Expensive shipping is a likely barrier to growth outside the South-East.

## Q5. How many customers come back?

In [11]:
repeat = run_sql_file(engine, "05_repeat_customers.sql")
show_query("05_repeat_customers.sql")
repeat

```sql
-- Q5. How many customers come back? (customer = customer_unique_id)
WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS orders
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT
    COUNT(*)                                              AS customers,
    COUNT(*) FILTER (WHERE orders >= 2)                   AS repeat_customers,
    ROUND(100.0 * COUNT(*) FILTER (WHERE orders >= 2) / COUNT(*), 2) AS repeat_rate_pct,
    ROUND(AVG(orders), 3)                                 AS avg_orders_per_customer,
    MAX(orders)                                           AS max_orders
FROM customer_orders;

```

,customers,repeat_customers,repeat_rate_pct,avg_orders_per_customer,max_orders
0,93350,2801,3.00,1.03,15


**Insights**
- Only **2,801 of 93,350 customers (3.0%) placed a second order**.
- The average customer makes 1.03 orders: Olist acquires customers but almost never keeps them.
- This is the most important finding of the project: growth so far was driven by acquisition alone.

## Q6. Cohort retention

In [12]:
cohorts = run_sql_file(engine, "06_cohorts.sql")
show_query("06_cohorts.sql")
cohorts

```sql
-- Q6. Monthly cohort retention
-- Cohort = month of a customer's first purchase.
-- month_number = months between the cohort month and the purchase month.
WITH purchases AS (
    SELECT
        c.customer_unique_id,
        DATE_TRUNC('month', o.order_purchase_timestamp)::date AS order_month
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
),
with_cohort AS (
    SELECT
        customer_unique_id,
        order_month,
        MIN(order_month) OVER (PARTITION BY customer_unique_id) AS cohort_month
    FROM purchases
),
activity AS (
    SELECT DISTINCT
        customer_unique_id,
        cohort_month,
        (EXTRACT(YEAR  FROM AGE(order_month, cohort_month)) * 12
       + EXTRACT(MONTH FROM AGE(order_month, cohort_month)))::int AS month_number
    FROM with_cohort
),
cohort_size AS (
    SELECT cohort_month, COUNT(DISTINCT customer_unique_id) AS customers
    FROM activity
    WHERE month_number = 0
    GROUP BY cohort_month
)
SELECT
    a.cohort_month,
    s.customers                                                AS cohort_size,
    a.month_number,
    COUNT(DISTINCT a.customer_unique_id)                       AS active_customers,
    ROUND(100.0 * COUNT(DISTINCT a.customer_unique_id) / s.customers, 2) AS retention_pct
FROM activity a
JOIN cohort_size s ON s.cohort_month = a.cohort_month
WHERE a.cohort_month BETWEEN '2017-01-01' AND '2018-08-01'
GROUP BY a.cohort_month, s.customers, a.month_number
ORDER BY a.cohort_month, a.month_number;

```

,cohort_month,cohort_size,month_number,active_customers,retention_pct
0,2017-01-01,717,0,717,100.00
1,2017-01-01,717,1,2,0.28
2,2017-01-01,717,2,2,0.28
3,2017-01-01,717,3,1,0.14
4,2017-01-01,717,4,3,0.42
...,...,...,...,...,...
202,2018-06-01,5875,1,25,0.43
203,2018-06-01,5875,2,16,0.27
204,2018-07-01,5946,0,5946,100.00
205,2018-07-01,5946,1,31,0.52


In [13]:
retention = cohorts.pivot(index="cohort_month", columns="month_number", values="retention_pct")
retention.index = pd.to_datetime(retention.index)

# A cohort can only be observed until the last month of data (Aug 2018).
# Empty cells inside that window mean 0% retention; cells after it are unknown and stay empty.
last_month = retention.index.max()
for cohort in retention.index:
    observable = (last_month.year - cohort.year) * 12 + (last_month.month - cohort.month)
    retention.loc[cohort, retention.columns <= observable] = retention.loc[cohort, retention.columns <= observable].fillna(0)

retention.index = retention.index.strftime("%Y-%m")
retention.iloc[:, :7]

month_number,0,1,2,3,4,5,6
cohort_month,,,,,,,
2017-01,100.00,0.28,0.28,0.14,0.42,0.14,0.42
2017-02,100.00,0.18,0.31,0.12,0.43,0.12,0.25
2017-03,100.00,0.44,0.36,0.40,0.36,0.16,0.16
2017-04,100.00,0.62,0.22,0.18,0.27,0.27,0.35
2017-05,100.00,0.46,0.46,0.29,0.29,0.32,0.41
2017-06,100.00,0.49,0.40,0.43,0.30,0.40,0.36
2017-07,100.00,0.53,0.35,0.24,0.29,0.21,0.32
2017-08,100.00,0.69,0.35,0.27,0.35,0.52,0.30
2017-09,100.00,0.70,0.55,0.27,0.45,0.22,0.22


In [14]:
m1 = cohorts[cohorts["month_number"] == 1]
print(f"Average month-1 retention: {m1['retention_pct'].mean():.2f}%")
print(f"Best month-1 retention:    {m1['retention_pct'].max():.2f}%")

Average month-1 retention: 0.48%
Best month-1 retention:    0.72%


**Insights**
- Month-1 retention is **below 1% for every cohort**: almost nobody buys again a month later.
- There is no visible improvement over time, so the problem is structural, not a one-off.
- The heatmap in `04_metrics.ipynb` will visualise this table.

## Q7. RFM scores

In [15]:
rfm = run_sql_file(engine, "07_rfm_base.sql")
show_query("07_rfm_base.sql")
print(f"{len(rfm):,} customers scored")
rfm.head()

```sql
-- Q7. RFM scores per customer with NTILE(5)
-- Recency   = days since the last purchase (reference = day after the last order in data)
-- Frequency = number of orders
-- Monetary  = total revenue
WITH order_revenue AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        o.order_purchase_timestamp,
        SUM(oi.price + oi.freight_value) AS revenue
    FROM orders o
    JOIN customers   c  ON c.customer_id = o.customer_id
    JOIN order_items oi ON oi.order_id   = o.order_id
    GROUP BY o.order_id, c.customer_unique_id, o.order_purchase_timestamp
),
reference AS (
    SELECT MAX(order_purchase_timestamp)::date + 1 AS ref_date FROM orders
),
rfm AS (
    SELECT
        r.customer_unique_id,
        (SELECT ref_date FROM reference) - MAX(r.order_purchase_timestamp)::date AS recency_days,
        COUNT(*)                     AS frequency,
        ROUND(SUM(r.revenue), 2)     AS monetary
    FROM order_revenue r
    GROUP BY r.customer_unique_id
)
SELECT
    customer_unique_id,
    recency_days,
    frequency,
    monetary,
    NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,   -- 5 = most recent
    NTILE(5) OVER (ORDER BY monetary ASC)      AS m_score,   -- 5 = highest spend
    CASE WHEN frequency >= 3 THEN 3                          -- frequency is too skewed
         WHEN frequency = 2  THEN 2                          -- for quintiles (97% buy once)
         ELSE 1 END                            AS f_score
FROM rfm;

```

93,350 customers scored


,customer_unique_id,recency_days,frequency,monetary,r_score,m_score,f_score
0,830d5b7aaa3b6f1e9ad63703bec97d23,714,1,143.46,1,4,1
1,7390ed59fa1febbfda31a80b4318c8cb,696,1,154.57,1,4,1
2,8d3a54507421dbd2ce0a1d58046826e0,696,1,133.46,1,4,1
3,10e89fd8e5c745f81bec101207ba4d7d,696,1,92.27,1,3,1
4,2f64e403852e6893ae37485d5fcacdaf,696,1,39.09,1,1,1


In [16]:
rfm[["recency_days", "frequency", "monetary"]].describe().T

,count,mean,std,min,25%,50%,75%,max
recency_days,"93,350.00",238.49,152.59,1.00,115.00,220.00,347.00,714.00
frequency,"93,350.00",1.03,0.21,1.00,1.00,1.00,1.00,15.00
monetary,"93,350.00",165.17,226.30,9.59,63.01,107.78,182.50,"13,664.08"


In [17]:
rfm["f_score"].value_counts().sort_index().rename("customers").to_frame()

,customers
f_score,
1,90549
2,2573
3,228


**Insights**
- Frequency cannot be split into quintiles: 97% of customers have exactly one order, so `NTILE` would put identical
  customers into different groups. A simple 1 / 2 / 3+ orders score is used instead. This is a deliberate methodological choice.
- Median spend per customer is ~108 BRL, but the distribution has a long tail of high-value customers.
- The segments (Champions, Loyal, At risk, etc.) are built from these scores in `04_metrics.ipynb`.

## Q8. Does delivery speed affect the review score?

In [18]:
delivery = run_sql_file(engine, "08_delivery_vs_review.sql")
show_query("08_delivery_vs_review.sql")
delivery

```sql
-- Q8. Does delivery speed affect the review score?
SELECT
    CASE
        WHEN o.delay_days <= -10 THEN '1. 10+ days early'
        WHEN o.delay_days <=   0 THEN '2. 0-9 days early'
        WHEN o.delay_days <=   5 THEN '3. 1-5 days late'
        WHEN o.delay_days <=  15 THEN '4. 6-15 days late'
        ELSE                          '5. 15+ days late'
    END                                                         AS delivery_vs_promise,
    COUNT(*)                                                    AS orders,
    ROUND(AVG(r.review_score), 2)                               AS avg_score,
    ROUND(100.0 * COUNT(*) FILTER (WHERE r.review_score <= 2) / COUNT(*), 1) AS low_score_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE r.review_score = 5)  / COUNT(*), 1) AS five_star_pct
FROM orders o
JOIN order_reviews r ON r.order_id = o.order_id
GROUP BY delivery_vs_promise
ORDER BY delivery_vs_promise;

```

,delivery_vs_promise,orders,avg_score,low_score_pct,five_star_pct
0,1. 10+ days early,61523,4.32,8.90,64.00
1,2. 0-9 days early,27920,4.22,10.10,58.50
2,3. 1-5 days late,2722,2.99,41.10,28.30
3,4. 6-15 days late,2479,1.74,78.20,7.90
4,5. 15+ days late,1180,1.73,78.30,7.50


In [19]:
late = pd.read_sql(
    """
    SELECT o.is_late,
           COUNT(*)                      AS orders,
           ROUND(AVG(r.review_score), 2) AS avg_score
    FROM orders o
    JOIN order_reviews r ON r.order_id = o.order_id
    GROUP BY o.is_late
    """,
    engine,
)
late

,is_late,orders,avg_score
0,False,89443,4.29
1,True,6381,2.27


**Insights**
- **Delivery is the strongest driver of satisfaction in the data.** On-time orders average 4.3 stars vs. 2.3 for late ones;
  orders more than 5 days late average **1.7 stars**, and 78% of them get 1–2 stars.
- Even 1–5 days of delay drops the score to ~3.0.
- Only ~7% of orders are late, but they produce a large share of negative reviews. Improving delivery
  for these orders is a direct lever on reviews and, likely, on repeat purchases.

## Q9. Cumulative revenue and year-over-year growth

In [20]:
running = run_sql_file(engine, "09_running_total.sql")
show_query("09_running_total.sql")
running

```sql
-- Q9. Cumulative revenue and year-over-year comparison by month
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp)::date AS month,
        SUM(oi.price + oi.freight_value)                       AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY month
)
SELECT
    month,
    ROUND(revenue, 2)                                                AS revenue,
    ROUND(SUM(revenue) OVER (ORDER BY month), 2)                     AS cumulative_revenue,
    ROUND(SUM(revenue) OVER (PARTITION BY EXTRACT(YEAR FROM month)
                             ORDER BY month), 2)                     AS ytd_revenue,
    ROUND(100.0 * (revenue - LAG(revenue, 12) OVER (ORDER BY month))
          / LAG(revenue, 12) OVER (ORDER BY month), 1)               AS yoy_growth_pct
FROM monthly
ORDER BY month;

```

,month,revenue,cumulative_revenue,ytd_revenue,yoy_growth_pct
0,2017-01-01,"127,482.37","127,482.37","127,482.37",NaN
1,2017-02-01,"271,239.32","398,721.69","398,721.69",NaN
2,2017-03-01,"414,330.95","813,052.64","813,052.64",NaN
3,2017-04-01,"390,812.40","1,203,865.04","1,203,865.04",NaN
4,2017-05-01,"566,657.40","1,770,522.44","1,770,522.44",NaN
5,2017-06-01,"490,050.37","2,260,572.81","2,260,572.81",NaN
6,2017-07-01,"566,299.08","2,826,871.89","2,826,871.89",NaN
7,2017-08-01,"645,832.36","3,472,704.25","3,472,704.25",NaN
8,2017-09-01,"701,077.49","4,173,781.74","4,173,781.74",NaN
9,2017-10-01,"751,117.01","4,924,898.75","4,924,898.75",NaN


**Insights**
- Cumulative revenue for the period is ~15.4 M BRL.
- Year-over-year growth falls from +745% (Jan 2018 vs. a tiny Jan 2017) to **+53% in Aug 2018**.
  YoY is still positive, but the slowdown confirms the plateau seen in Q1–Q2.

## Q10. Payment methods

In [21]:
payments = run_sql_file(engine, "10_payment_types.sql")
show_query("10_payment_types.sql")
payments

```sql
-- Q10. Payment methods: share, typical amount and installments
SELECT
    payment_type,
    COUNT(DISTINCT order_id)                                        AS orders,
    ROUND(100.0 * COUNT(DISTINCT order_id)
          / (SELECT COUNT(DISTINCT order_id) FROM order_payments), 1) AS order_share_pct,
    ROUND(100.0 * SUM(payment_value) / SUM(SUM(payment_value)) OVER (), 1) AS value_share_pct,
    ROUND(AVG(payment_value), 2)                                    AS avg_payment,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY payment_value)::numeric, 2) AS median_payment,
    ROUND(AVG(payment_installments), 1)                             AS avg_installments
FROM order_payments
GROUP BY payment_type
ORDER BY orders DESC;

```

,payment_type,orders,order_share_pct,value_share_pct,avg_payment,median_payment,avg_installments
0,credit_card,74297,77.00,78.50,162.25,106.81,3.50
1,boleto,19191,19.90,18.00,144.33,93.78,1.00
2,voucher,3679,3.80,2.20,62.45,39.04,1.00
3,debit_card,1484,1.50,1.30,140.11,89.58,1.00


**Insights**
- **Credit cards dominate**: 77% of orders and 78.5% of payment value, with 3.5 installments on average.
  Buying in installments is a core part of Brazilian e-commerce.
- `boleto` (bank slip, paid in cash) covers ~20% of orders: an important option for customers without cards.
- Vouchers are mostly small partial payments (median 39 BRL).

## Summary of SQL findings

| Question | Answer |
| --- | --- |
| Revenue trend | Fast growth in 2017, plateau at ~1.0–1.1 M BRL/month from Mar 2018 |
| Growth driver | More orders; AOV stable at ~150–170 BRL |
| Top categories | 5 of 74 categories = 39% of revenue (health & beauty, watches & gifts, bed & bath) |
| Geography | SP + RJ + MG = 63% of revenue; remote states pay up to 22% freight |
| Retention | Only 3.0% of customers buy twice; month-1 cohort retention < 1% |
| Satisfaction | Average score 4.3 on time vs. 2.3 late; 1.7 when 5+ days late |
| Payments | 77% credit card, 3.5 installments on average |

**Main business conclusion:** Olist grew by acquiring new customers, but it almost never converts them into
repeat buyers, and growth has stalled. The clearest lever found so far is delivery reliability:
late orders destroy satisfaction.

Next step → `04_metrics.ipynb`: RFM segments, cohort heatmap and the charts for the final report.